# Solutions — Conditional rendering & lists

Only look here after you've actually tried the exercises in `conditional_lists.ipynb`.

### LESSON 18 — Exercise

The whole exercise turns on one fact: `&&` hands back the **left value** when it is falsy,
so the question is never "is it falsy" but "does React render it".

In [ ]:
function l18guard(left) {
  const result = left && "<Badge/>";

  if (result === "<Badge/>") return "badge";
  if (result === null || result === undefined) return "nothing";
  if (typeof result === "boolean") return "nothing";
  if (result === "") return "nothing";
  return `LEAK: ${String(result)}`;
}

for (const left of [false, null, undefined, "", 0, NaN, 3, "hello"]) {
  console.log(String(left).padEnd(10), "->", l18guard(left));
}

**Part 2 — the boolean guard.** `count > 0` is a comparison, and a comparison is always
`true` or `false`. Both render nothing when false, so there is nothing left to leak —
including for `NaN`, where every comparison is `false`.

In [ ]:
const l18carts = [{ count: 0 }, { count: 3 }, { count: NaN }];

for (const cart of l18carts) {
  const guard = cart.count > 0;
  console.log(
    `count=${String(cart.count).padEnd(4)}`,
    `count > 0 is ${String(guard).padEnd(5)}`,
    "->", l18guard(guard),
  );
}

**Common mistakes.**

- Reporting `""` as a leak. The empty string *is* returned by `&&`, but React renders it as
  nothing visible — so the user sees no difference. It is the two **numbers** that show.
- Testing with `if (!result)` to decide "nothing". That is the same falsy reflex the lesson
  is about: `0` and `NaN` are falsy *and* render. The test has to be "which values does
  React refuse to render", not "which values are falsy".
- Assuming `NaN > 0` throws or is somehow special. It is simply `false` — every comparison
  with `NaN` is `false`, which is exactly why the boolean guard rescues it too.

### LESSON 18 — Mini challenge

**Part 1 — what the playground shows.** Two lines, same data:

```text
Broken guard (count=0): 0
Boolean guard (count=0):
```

The first is the bug on screen: a bare `0` where the designer expected nothing. At the
bottom, the six falsy values line up with the lesson's table — `false`, `null`, `undefined`
and `""` leave nothing behind, while `zero: 0` and `NaN: NaN` both print.

**Part 2 — choosing the shape.**

1. **`Price` renders nothing when unavailable — early return.** The component has no tree to
   draw at all, so `if (!available) return null;` says exactly that, once, at the top.
2. **Featured badge inside a card that always renders — `&&`.** The card, its heading and
   its layout render either way; only this one element comes and goes.
3. **`Online` / `Offline` — ternary.** Two outcomes, both of them something. `&&` cannot
   express this, because its false branch is always "nothing".
4. **Summary versus an access-denied screen — early return.** Two completely different
   trees. A ternary wrapping the entire body would work and would be much harder to read —
   the early return puts the exceptional case first and leaves the normal one unindented.

**Part 3 — the two questions.**

*What the user sees, and two fixes.* With no unread messages, `user.unreadCount` is `0`, so
`{user.unreadCount && <Dot />}` evaluates to `0` and a stray **0** appears where the dot
would have been. Two fixes:

```jsx
{user.unreadCount > 0 && <Dot />}        // a boolean on the left
{user.unreadCount > 0 ? <Dot /> : null}  // or choose both outcomes explicitly
```

A third, sometimes the tidiest, is to compute it above the `return`:

```jsx
const hasUnread = user.unreadCount > 0;
```

which also gives the condition a name, and names are worth more than they cost.

*Why a ternary can never leak.* Because you write both outcomes yourself. `cond` is only
ever used to **choose** between them — it is never the value that gets rendered. With `&&`,
the left-hand value is the fallback, so whatever it happens to be lands in your JSX. The
ternary has no fallback to leak.

That is the real distinction, and it is worth keeping: `&&` renders the condition when the
condition is false; a ternary renders what you told it to.

### LESSON 19 — Exercise

In [ ]:
const l19cart = [
  { name: "Bread", price: 2.4 },
  { name: "Milk", price: 1.15 },
  { name: "Eggs", price: 3.8 },
];

// Part 1 — one item in, one string out.
function l19rows(cart) {
  return cart.map((item) => `${item.name} — ${item.price.toFixed(2)} EUR`);
}

console.log(l19rows(l19cart));
console.log("length:", l19rows(l19cart).length);

// Part 2 — a block body with no return.
function l19rowsBroken(cart) {
  return cart.map((item) => {
    `${item.name} — ${item.price.toFixed(2)} EUR`;
  });
}

console.log(l19rowsBroken(l19cart));

// React would render ZERO <li> elements for this one. The array still has three items,
// but every item is undefined, and undefined renders nothing (LESSON 10). Nothing is
// logged as an error because nothing went wrong as far as JavaScript is concerned —
// a function with no return statement returns undefined, which is perfectly legal.

// Part 3 — .forEach() returns undefined. It is for doing something with each item, not
// for producing a new array, so there would be no array of elements to render at all.

**Common mistakes.**

- `cart.map((item) => { return item.name + " — " + item.price })` without `.toFixed(2)`.
  `1.15` prints as `1.15`, but `2.4` prints as `2.4` rather than `2.40`, so the column does
  not line up. Formatting is not decoration here — it is the difference between a price list
  and a mess.
- Using `.forEach()` and pushing into an array by hand. It works, and it is three lines
  where one would do. The reason `.map()` exists is exactly this shape.
- Expecting the broken version to throw. It cannot: returning nothing from a function is
  normal JavaScript. The failure surfaces as an empty list in the browser, which is why this
  bug is usually found by eye rather than by the console.

### LESSON 19 — Mini challenge

**Part 1 — what the playground shows.** The first two lists render three rows each; the
third renders none, because its callback has a block body and no `return`. The console
carries one warning:

```text
Each child in a list should have a unique "key" prop.
Check the render method of `Experiment04`.
```

React is telling you something real, and LESSON 20 is the answer. Note that the warning
fires for the two lists that *did* render — the empty one produced no elements to complain
about.

**Part 2 — reasoning.**

1. **LESSON 10**, the what-renders table: *an array renders each item, in order*. That row
   is the entire reason `.map()` works in JSX. `.map()` produces an array; React already
   knew what to do with one.

2. **`.map()` is not a React thing.** It is `Array.prototype.map`, standard JavaScript, and
   it behaves identically outside React — as the example cell shows, with no React imported
   anywhere. React contributes only the second half: it renders the array it is handed.

3. **`rows` is an array of element objects — plain data.** Nothing has been rendered.
   LESSON 7 established that an element is a description, not a DOM node, and that writing
   JSX builds a value rather than putting something on screen. Storing that array in a
   variable, logging it, or passing it around changes nothing; rendering happens only when
   React receives it, in the `return`.

   This is also why `{rows}` works in the JSX afterwards: it is the same array either way.

### LESSON 20 — Exercise

In [ ]:
const l20products = [
  { id: "p1", sku: "BRD-01", name: "Bread", price: 2.4 },
  { id: "p2", sku: "MLK-01", name: "Milk", price: 1.15 },
  { id: "p3", sku: "EGG-01", name: "Eggs", price: 3.8 },
  { id: "p4", sku: "BRD-01", name: "Bread (large)", price: 3.9 },
];

function l20isUsableKey(items, pick) {
  const keys = items.map(pick);
  const unique = new Set(keys).size === keys.length;
  const defined = keys.every((k) => k !== undefined && k !== null);
  return { keys, unique, defined, usable: unique && defined };
}

// Part 1
for (const [label, pick] of [
  ["id", (p) => p.id],
  ["sku", (p) => p.sku],
  ["name", (p) => p.name],
  ["price", (p) => p.price],
]) {
  const result = l20isUsableKey(l20products, pick);
  console.log(label.padEnd(6), "usable:", result.usable, "|", JSON.stringify(result.keys));
}

// `sku` is the one that fails: BRD-01 appears twice, because the shop sells two sizes of
// the same bread. It LOOKS like an identifier and is not unique among these siblings.
//
// `name` and `price` PASS the check - in this snapshot they happen to be unique. They are
// still bad keys, and that is the point of the second question: the check can only look at
// one render of one array. It cannot see that a second "Bread" could be added tomorrow, or
// that a price can be edited - and a key that changes when the data changes is not identity,
// it is a coincidence. Only `id` is a value that belongs to the item and never moves.

**Part 2 — what the removal does to each key.**

In [ ]:
const l20after = l20products.slice(1);

function l20trace(before, after, pick) {
  return after.map((item) => {
    const wasIndex = before.indexOf(item);
    const nowIndex = after.indexOf(item);
    const was = pick(item, wasIndex);
    const now = pick(item, nowIndex);
    return `${item.name}: key was ${was}, key is now ${now}`;
  });
}

console.log("--- keyed by id ---");
for (const line of l20trace(l20products, l20after, (p) => p.id)) console.log(" ", line);

console.log("--- keyed by index ---");
for (const line of l20trace(l20products, l20after, (p, i) => i)) console.log(" ", line);

The id keys are identical before and after: `p2` is `p2` whatever happens to the array. The
index keys all shift down by one, so every surviving item now claims the identity of the item
that used to sit above it — and React believes it.

**Part 3 — why `Math.random()` is the worst key.** `l20isUsableKey` only checks one render.
It cannot see the rule that matters most here: **keys must not change between renders.** A
random key is unique *and* different every single time, so nothing in the new list matches
anything in the old one. React concludes every item was deleted and a brand-new set inserted,
throws away all the DOM and rebuilds it — losing any user input inside those rows, every
render. The index at least stays the same for an unchanged list; a random key is wrong even
when nothing changed.

**Common mistakes.**

- Concluding `sku` is fine because it is called a code. Uniqueness is a property of *this
  list*, not of the field's name. Check it, do not assume it.
- Writing `key={i}` inside the callback and thinking the rule is about the letter `i`. The
  problem is not the variable - it is that the value describes a position.
- Using `JSON.stringify(item)` as a key. It is stable and unique right up to the moment
  someone edits an item, at which point that row loses its identity and its DOM.

### LESSON 20 — Mini challenge

**Part 1 — what the experiment shows.** Type notes into every row of both lists, then delete
the first product and save. The result, measured:

```text
keyed by index                  keyed by id
Milk  => "note-BREAD"           Milk  => "note-MILK"
Eggs  => "note-MILK"            Eggs  => "note-EGGS"
```

**3.** In the index-keyed list, **Milk** now holds the note you typed for Bread, and the note
you typed for Eggs is gone entirely. Each note moved up one row: React kept the DOM node at
position 0 and simply relabelled it, because you told it position 0 *was* the identity.

**4. Why the console stayed silent.** Both lists had keys, so the warning from LESSON 19
never appeared. That warning only checks that a key is *present*. Nothing checks that it is
the *right* key, and nothing can — only you know which field identifies an item. This is the
real lesson: silencing the warning and fixing the problem are two different jobs.

**Part 2 — reasoning.**

1. **They fixed the warning, not the bug.** `key={index}` satisfies React's "please give me a
   key" check while telling React exactly the thing that was already wrong: identify items by
   position. The console goes quiet and the identity mismatch survives — arguably worse than
   before, because the one signal that something needed attention is now gone.

2. **No, `key={index}` is not wrong there — and saying otherwise is over-correcting.** Append
   only, no reordering, no removal from the middle, no state inside the rows: the index is
   stable for every item that exists, which is the actual requirement. It is worth knowing
   the rule rather than the superstition, because the day that list gains a delete button,
   the index stops qualifying and you need to notice.

3. **A stable key can come from** the data's own identifier — a database id, an order number,
   a slug — or from an id generated **once, where the item is created**, and stored with it.

   **It must not come from** anything computed during render: `Math.random()`, a counter that
   increments each render, the array index, or the item's position in a sorted view. If the
   value is invented while rendering, it is not identity — it is a coincidence that held for
   one render.

### LESSON 21 — Exercise

In [ ]:
const l21catalogue = [
  { id: "a1", name: "Bread", price: 2.4, inStock: true },
  { id: "a2", name: "Milk", price: 1.15, inStock: false },
  { id: "a3", name: "Eggs", price: 3.8, inStock: true },
  { id: "a4", name: "Butter", price: 2.95, inStock: true },
];

function l21view(catalogue) {
  const inStock = catalogue.filter((item) => item.inStock);
  const sorted = [...inStock].sort((a, b) => a.price - b.price);

  return {
    shown: sorted.map((item) => item.name),
    total: catalogue.length,
    hidden: catalogue.length - inStock.length,
  };
}

console.log(l21view(l21catalogue));
console.log("original order:", l21catalogue.map((item) => item.name));

The copy in `[...inStock].sort(...)` is belt and braces here — `.filter()` already gave a
new array, so sorting it harms nothing. Keeping the spread anyway is a good habit: the day
someone deletes the filter, the sort is still safe.

**Part 2 — the empty-state message.**

In [ ]:
function l21emptyMessage(shown, query) {
  if (shown.length === 0) return `No products match "${query}".`;
  return null;
}

console.log(l21emptyMessage([], "kiwi"));
console.log(l21emptyMessage(["Bread"], "bre"));

// Part 3 — a stored `hiddenCount` is a second source of truth. The moment an item is
// added, removed or goes out of stock, there are two things to update and only one of them
// is obvious. When they disagree the screen lies: "1 hidden" beside two hidden rows, and
// nothing errors. A value recalculated from the array cannot disagree with the array.

**Common mistakes.**

- Sorting before filtering. It works, and it sorts items you are about to throw away. Filter
  first, then sort what is left.
- `catalogue.filter(...).sort(...)` with no copy. Also safe, for the reason above — but only
  because `.filter()` created the array. `catalogue.sort(...)` directly is the bug.
- Computing `hidden` as `shown.length - total`, which is negative. It is
  `total - inStock.length`.
- Returning `""` instead of `null` from `l21emptyMessage` when there is nothing to say.
  Both are falsy, but `null` says "no message" while `""` says "a message, which is empty".

### LESSON 21 — Mini challenge

**1. The bug.** `products.sort(...)` does not return a sorted copy — it sorts `products`
itself and returns that same array. `products` arrived as a **prop**, so the component has
just edited data belonging to its parent (LESSON 14: props are read-only). Both lists render
from the same now-reordered array, so the "original order" list is gone — and the parent's
copy is reordered too, for every other component using it.

The fix is one character short of trivial:

```js
const byPrice = [...products].sort((a, b) => a.price - b.price);
```

**2. What the user sees.** When nothing matches, `shown.length` is `0`. `&&` hands back its
left value when that value is falsy (LESSON 18), so the expression evaluates to `0`, and
React renders a bare **0** where the results line should be. The fix is the same as it was
then — put a comparison on the left:

```jsx
{shown.length > 0 && <p>{shown.length} results</p>}
```

This is the `0` trap in its natural habitat: counts are exactly where it bites.

**3. The argument against storing it.** Correctness, not speed. A stored `hiddenCount` is a
second copy of a fact the array already contains, and two copies of one fact will eventually
disagree. Every place that changes the catalogue now has a second obligation, nothing
enforces it, and the failure is silent — the number is simply wrong, and it looks exactly
like a number that is right.

A derived value has none of that. It cannot be forgotten, because it is recalculated from
the only thing that is true.